# Day 25 — What makes a good eval

You've written scoring functions since Day 06. Today: the theory that makes them trustworthy —
the dimensions worth measuring, how to evaluate when you have **no ground-truth labels**, how
to build and calibrate an **LLM-as-judge**, and why offline evals and production monitoring are
different jobs.

## Agenda (60 min)

| # | Segment | Time |
| - | ------- | ---- |
| 0 | An eval is a measurement instrument | 3 min |
| 1 | The dimensions: accuracy, relevance, faithfulness, latency, cost | 10 min |
| 2 | Reference-based vs reference-free | 10 min |
| 3 | LLM-as-judge: build one, then break it | 16 min |
| 4 | Evaluating with no ground truth | 12 min |
| 5 | Offline eval vs production monitoring | 6 min |
| 6 | Exercises + quiz | 3 min |

Kernel: **Python (ai-upskill)**.

In [1]:
import numpy as np, re, json
from sentence_transformers import SentenceTransformer
emb = SentenceTransformer("all-MiniLM-L6-v2")
def E(x): return emb.encode(x if isinstance(x, list) else [x], normalize_embeddings=True)
rng = np.random.default_rng(0)
print("ready")

/Users/umeshkaranam/Desktop/personal/UPSKILL/AI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8620.10it/s]

ready


## 0 — An eval is a measurement instrument (3 min)

A good instrument is **valid** (measures the thing you care about), **reliable** (same input →
same score), **sensitive** (moves when quality moves), and **cheap enough to run often**. A
bad eval is worse than none: it gives false confidence and you optimise toward its blind spots
(Goodhart — Day 10).

The eval's job is to answer one question fast: *did this change make the product better?*

## 1 — The dimensions (10 min)

| Dimension | Question | How to measure |
| --------- | -------- | -------------- |
| **Accuracy / correctness** | is the answer right? | exact match / F1 vs a reference; LLM-judge; task-specific check |
| **Relevance** | does it address *this* question? | embedding similarity(answer, question); judge |
| **Faithfulness / groundedness** | is every claim supported by the provided context? | per-claim entailment check; judge (RAG-critical) |
| **Completeness** | did it cover all parts of the question? | checklist of required elements present |
| **Format validity** | does it parse / match the schema? | a parser or JSON-schema validator (Day 23) |
| **Consistency** | same answer on re-runs / paraphrases? | variance across N samples |
| **Safety** | refuses what it should, no PII leak, no toxic output | classifier + rule checks |
| **Latency** | p50 / p95 seconds | measure |
| **Cost** | $ per call / per task | tokens × price |

**Correctness alone is not enough.** A RAG answer can be correct *and* unfaithful (Day 18) —
the model knew it, the retriever didn't surface it, and next time the model's prior is wrong.

In [2]:
QUESTION = "How long do refunds take and do I need the original receipt?"
CONTEXT = ["Refunds are processed within 5 business days.",
           "A receipt is not required for refunds under $50."]
ANSWER_GOOD = "Refunds take 5 business days, and you don't need a receipt for refunds under $50."
ANSWER_UNFAITHFUL = "Refunds take 5 business days. You must mail the original receipt to our office."
ANSWER_IRRELEVANT = "Our support team is available Monday to Friday, 9am to 5pm Pacific."

def relevance(answer, question):
    return float(E(answer)[0] @ E(question)[0])

def faithfulness(answer, context):
    claims = [c.strip() for c in re.split(r"(?<=[.!?])\s+", answer) if len(c.strip()) > 12]
    ctx_vecs = E(context)
    supported = []
    for c in claims:
        best = float(np.max(ctx_vecs @ E(c)[0]))
        supported.append(best >= 0.55)          # 0.55 ~ "this claim is entailed by some passage"
    return np.mean(supported), list(zip(claims, supported))

for name, a in [("good", ANSWER_GOOD), ("unfaithful", ANSWER_UNFAITHFUL), ("irrelevant", ANSWER_IRRELEVANT)]:
    f, detail = faithfulness(a, CONTEXT)
    print(f"{name:11s} relevance={relevance(a, QUESTION):.2f}  faithfulness={f:.2f}")
    for claim, ok in detail:
        print(f"    [{'OK ' if ok else 'XX '}] {claim}")

good        relevance=0.82  faithfulness=1.00
    [OK ] Refunds take 5 business days, and you don't need a receipt for refunds under $50.
unfaithful  relevance=0.87  faithfulness=0.50
    [OK ] Refunds take 5 business days.
    [XX ] You must mail the original receipt to our office.
irrelevant  relevance=0.05  faithfulness=0.00
    [XX ] Our support team is available Monday to Friday, 9am to 5pm Pacific.


The embedding-threshold entailment check is a *proxy* (a real one uses an NLI model or an
LLM-judge), but it already separates the three answers: the unfaithful one has a claim no
passage supports; the irrelevant one scores low on relevance to the question.

## 2 — Reference-based vs reference-free (10 min)

**Reference-based**: compare output to a gold answer.

| Metric | Good for | Breaks on |
| ------ | -------- | --------- |
| Exact match | short, canonical answers (a number, a label) | any valid paraphrase → scored wrong |
| Token F1 | short-answer QA | word overlap ≠ meaning |
| BLEU / ROUGE | translation / summarisation *n*-gram overlap | fluent correct paraphrase scores low; gibberish with right words scores high |
| Embedding similarity | "is it about the same thing" | can't tell *correct* from *plausible-but-wrong* |

**Reference-free**: no gold answer — score properties of the output.

- faithfulness / groundedness (claims ⊆ context)
- relevance (answer ↔ question)
- format validity, safety, self-consistency
- LLM-as-judge against a rubric (§3)

In [3]:
def exact_match(pred, gold): return int(pred.strip().lower() == gold.strip().lower())
def token_f1(pred, gold):
    p, g = pred.lower().split(), gold.lower().split()
    common = sum((set(p) & set(g)).__contains__(w) for w in p)  # rough
    if not p or not g: return 0.0
    prec = len(set(p) & set(g)) / len(set(p)); rec = len(set(p) & set(g)) / len(set(g))
    return 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)

gold = "5 business days"
for pred in ["5 business days", "five business days", "about a week", "5 business days (to your card)"]:
    print(f"EM={exact_match(pred, gold)} F1={token_f1(pred, gold):.2f} sim={float(E(pred)[0]@E(gold)[0]):.2f}  <- {pred!r}")
print("\n-> 'five business days' is correct but EM=0. 'about a week' is close in meaning, wrong")
print("   in fact. No single metric is right; combine, and prefer judge/faithfulness for prose.")

EM=1 F1=1.00 sim=1.00  <- '5 business days'
EM=0 F1=0.67 sim=0.92  <- 'five business days'
EM=0 F1=0.00 sim=0.57  <- 'about a week'
EM=0 F1=0.67 sim=0.76  <- '5 business days (to your card)'

-> 'five business days' is correct but EM=0. 'about a week' is close in meaning, wrong
   in fact. No single metric is right; combine, and prefer judge/faithfulness for prose.


## 3 — LLM-as-judge: build one, then break it (16 min)

An LLM-judge is a model prompted with a **rubric** to score another model's output. It's the
workhorse for prose quality where reference metrics fail. Here's the shape (real judge would be
`claude-opus-5`); we use a deterministic mock judge with **injectable biases** to study the
failure modes.

In [4]:
JUDGE_PROMPT = '''You are grading an answer. Score 1-5 on: correctness, relevance, faithfulness
to the provided context. Output JSON: {"score": <1-5>, "reason": "<one sentence>"}.
Question: {q}
Context: {ctx}
Answer: {a}'''

def mock_judge(question, context, answer, *, bias=None):
    # a deterministic "judge": base score from real relevance + faithfulness signals...
    f, _ = faithfulness(answer, context)
    r = relevance(answer, question)
    base = 1 + 3.2 * (0.5 * f + 0.5 * min(1.0, r / 0.6))
    # ...plus a bias term to demonstrate known judge failure modes
    if bias == "verbosity":  base += 1.0 * (len(answer.split()) > 25)       # longer looks better
    if bias == "position_A": base += 0.5                                    # first-shown bias (pairwise)
    if bias == "self":       base += 0.7 * ("[our model]" in answer)        # self-preference
    return round(float(np.clip(base, 1, 5)), 2)

a_short = "5 business days; no receipt under $50."
a_long  = a_short + " Please note this applies to standard orders processed through our system and " \
                    "is measured from the date your refund request is approved by an agent."
print("unbiased judge, short:", mock_judge(QUESTION, CONTEXT, a_short))
print("unbiased judge, long :", mock_judge(QUESTION, CONTEXT, a_long))
print("verbosity-biased, long:", mock_judge(QUESTION, CONTEXT, a_long, bias="verbosity"),
      "  <- same facts, higher score just for length")

unbiased judge, short: 4.2
unbiased judge, long : 4.2
verbosity-biased, long: 5.0   <- same facts, higher score just for length


In [5]:
# Calibration: does the judge agree with human labels? Build a tiny labelled set and measure.
HUMAN_SET = [  # (question, context, answer, human_score 1-5)
 (QUESTION, CONTEXT, ANSWER_GOOD, 5),
 (QUESTION, CONTEXT, a_long, 4),
 (QUESTION, CONTEXT, ANSWER_UNFAITHFUL, 2),
 (QUESTION, CONTEXT, ANSWER_IRRELEVANT, 1),
 (QUESTION, CONTEXT, "Refunds take 5 business days.", 3),         # correct but incomplete
]
def eval_judge(bias=None):
    preds = [mock_judge(q, c, a, bias=bias) for q, c, a, _ in HUMAN_SET]
    humans = [h for *_, h in HUMAN_SET]
    mae = np.mean(np.abs(np.array(preds) - np.array(humans)))
    # rank correlation (Spearman-ish via argsort)
    corr = np.corrcoef(np.argsort(np.argsort(preds)), np.argsort(np.argsort(humans)))[0, 1]
    return mae, corr, preds, humans

for b in [None, "verbosity"]:
    mae, corr, p, h = eval_judge(b)
    print(f"bias={str(b):10s}  MAE vs human={mae:.2f}  rank-corr={corr:.2f}  preds={p} humans={h}")

bias=None        MAE vs human=0.75  rank-corr=0.60  preds=[4.2, 4.2, 3.4, 1.14, 4.2] humans=[5, 4, 2, 1, 3]


bias=verbosity   MAE vs human=0.91  rank-corr=0.70  preds=[4.2, 5.0, 3.4, 1.14, 4.2] humans=[5, 4, 2, 1, 3]


### Judge failure modes (all documented in the literature)

| Bias | The judge... | Mitigation |
| ---- | ------------ | ---------- |
| **Verbosity / length** | scores longer answers higher | tell it to ignore length; normalise; penalise padding in the rubric |
| **Position** (pairwise) | prefers the first (or a fixed) option | swap order and average; run both directions |
| **Self-preference** | prefers text from the same model family | use a different model as judge; ensemble |
| **Sycophancy** | agrees with assertive/confident tone | rubric emphasises evidence over confidence |
| **Format halo** | markdown / structure reads as quality | judge content only; strip formatting |

**Always calibrate the judge against human labels** on a sample (50–100). Report MAE and rank
correlation. If the judge doesn't correlate with humans, fix the rubric before trusting it at
scale. A judge you haven't calibrated is a random number generator with good PR.

## 4 — Evaluating with no ground truth (12 min)

Most real apps have no gold answers. You still can:

1. **Property checks** — faithfulness (claims ⊆ context), format validity, no-PII, on-topic.
   Cheap, deterministic, catch the scary failures.
2. **Self-consistency** — sample N answers; low agreement = low confidence. Works for anything
   with a discrete-ish answer.
3. **LLM-judge with a rubric** — calibrated (§3) on a small human-labelled slice.
4. **Pairwise vs a baseline** — "is the new version better than the old one on this input?" is
   easier for a judge than an absolute score, and it's exactly the question you care about.
5. **Human review of a sample** — 20–50 outputs per release, especially the disagreement set.

In [6]:
# Self-consistency as a confidence signal (no ground truth needed)
def sample_answers(question, n=7):
    # simulate a model that's confident on easy Qs, wobbly on hard ones
    easy = "refund" in question.lower()
    pool = (["5 business days"] * 6 + ["3-5 business days"]) if easy else \
           ["it depends", "around 10 days", "check your email", "2 weeks", "unclear", "7 days", "varies"]
    return list(rng.choice(pool, size=n))

for q in ["how long do refunds take", "what will my custom quote be"]:
    ans = sample_answers(q)
    from collections import Counter
    top, count = Counter(ans).most_common(1)[0]
    print(f"{q:38s} -> agreement {count}/{len(ans)} on {str(top)!r}  (low agreement = flag for review)")

how long do refunds take               -> agreement 7/7 on '5 business days'  (low agreement = flag for review)
what will my custom quote be           -> agreement 2/7 on 'unclear'  (low agreement = flag for review)


In [7]:
# Pairwise: "is B better than A?" -- easier and more actionable than an absolute score
def pairwise_judge(question, context, answer_a, answer_b):
    sa = 0.5 * faithfulness(answer_a, context)[0] + 0.5 * min(1, relevance(answer_a, question)/0.6)
    sb = 0.5 * faithfulness(answer_b, context)[0] + 0.5 * min(1, relevance(answer_b, question)/0.6)
    # mitigate position bias: this mock is symmetric, but a real judge you'd call twice (A,B) and (B,A)
    return "B" if sb > sa + 0.05 else ("A" if sa > sb + 0.05 else "tie")

print("good vs unfaithful   :", pairwise_judge(QUESTION, CONTEXT, ANSWER_GOOD, ANSWER_UNFAITHFUL))
print("incomplete vs good   :", pairwise_judge(QUESTION, CONTEXT, "Refunds take 5 business days.", ANSWER_GOOD))
print("\n-> ship the change if it wins > X% of pairwise comparisons on your eval set, "
      "with position-swapped double-runs.")

good vs unfaithful   : A
incomplete vs good   : tie

-> ship the change if it wins > X% of pairwise comparisons on your eval set, with position-swapped double-runs.


## 5 — Offline eval vs production monitoring (6 min)

| | **Offline eval** | **Production monitoring** |
| --- | --- | --- |
| When | before a change ships (CI, hill-climbing) | continuously, on live traffic |
| Data | a frozen, curated test set | real user inputs (unlabelled, shifting) |
| Question | "is the new version better?" | "is quality holding? anything on fire?" |
| Metrics | correctness, faithfulness, judge scores vs the set | latency/error rates, cost, guardrail hits, thumbs-down rate, judge-on-a-sample, drift |
| Ground truth | yes (that's why you curated it) | rarely — property checks + sampled judge + user signals |
| Failure it catches | regressions from your change | prompt drift, model updates, new query types, abuse, outages |

You need **both**. Offline eval keeps a frozen test split you never train/tune on. Production
monitoring feeds new failure cases *back* into the offline set (the disagreement set, the
thumbs-down transcripts) so the eval keeps up with reality. That loop is Day 26–27.

## 6 — Exercises

1. **Faithfulness threshold.** Sweep the 0.55 entailment threshold from 0.4 to 0.75 on the
   three answers. Plot faithfulness score vs threshold. Where does the unfaithful answer stop
   being flagged, and what does a too-low threshold cost you?
2. **Metric disagreement.** Build 10 (pred, gold) pairs where EM, F1, and embedding-sim
   disagree. Tabulate. Which metric would you trust for (a) a math answer, (b) a summary,
   (c) a yes/no?
3. **Position bias.** Extend `pairwise_judge` to a real-ish version that adds `+0.4` to
   whichever answer is passed first. Show that running (A,B) and (B,A) and requiring agreement
   removes the bias (ties when it flips).
4. **Calibrate a rubric change.** Add a "length penalty" line to `JUDGE_PROMPT` (in the mock,
   subtract `0.3 * (words > 40)`). Re-run `eval_judge`. Does MAE vs human improve?
5. **Self-consistency threshold.** For `sample_answers`, define "confident" as agreement ≥
   5/7. Over 50 simulated questions (mix easy/hard), what fraction get auto-answered vs routed
   to review? Tune the threshold.
6. **Build a mini eval report.** Given a list of (question, context, answer), output a table
   with relevance, faithfulness, format_ok (JSON parses), word count, and an overall
   pass/fail (faithfulness ≥ 0.8 AND relevance ≥ 0.3). This is the Day 26 harness in miniature.

> **Attempt every exercise and the quiz first.** The worked solutions and the answer key live in [`solutions/solutions.ipynb`](solutions/solutions.ipynb) — open it only to check your work, not to start.

## Self-check quiz


1. Name the four properties of a good measurement instrument.
2. Why is correctness alone insufficient for a RAG eval?
3. Give a case where BLEU/ROUGE scores a bad answer high and a good answer low.
4. Name three LLM-judge biases and a mitigation for each.
5. What must you do before trusting an LLM-judge at scale?
6. You have no ground-truth labels. Name three ways to evaluate anyway.
7. How do offline eval and production monitoring differ in data, question, and ground truth?

## Where this goes next

- **Day 26 — Build an eval harness:** turn today's dimensions into a runnable harness (10+
  test cases, scoring functions, a report) for the Week 6 RAG pipeline, and wire the
  disagreement set back into it.